In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

In [2]:
group_order = ["DSI", "PHQ", "GAD", "PCL", "PDSS", "AUDIT"]

group_to_label = {
    "DSI": "DSI_group",
    "PHQ": "PHQ_group",
    "GAD": "GAD_group",
    "PCL": "PCL_group",
    "PDSS": "PDSS_group",
    "AUDIT": "AUDIT_group",
}

In [3]:
X_lock = pd.read_csv("lockbox_X_final.csv")
Y_lock = pd.read_csv("lockbox_Y.csv")

print(X_lock.shape)
print(Y_lock.shape)
X_lock.head()

(241, 60)
(241, 6)


,no,DSI_1,DSI_2,DSI_3,DSI_4,PHQ_1,PHQ_2,PHQ_3,PHQ_4,PHQ_5,...,AUDIT_3,AUDIT_4,AUDIT_5,AUDIT_6,AUDIT_7,AUDIT_8,AUDIT_9,AUDIT_10,Age,Sex
0,2217,0,0,0,0,0,0,0,0,0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,29.0,1.0
1,806,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,73.0,1.0
2,1018,1,0,1,1,0,0,0,0,0,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.0,1.0
3,998,1,1,1,0,3,3,3,3,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,1.0
4,1398,1,1,1,1,2,1,2,1,2,...,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,26.0,1.0


In [4]:
models = {}
for g in group_order:
    models[g] = joblib.load(f"catboost_{g}_final.pkl")

print(models.keys())

dict_keys(['DSI', 'PHQ', 'GAD', 'PCL', 'PDSS', 'AUDIT'])


In [5]:
def safe_acc(y_true, y_score, thr=0.5):
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return np.nan
    y_pred = (y_score >= thr).astype(int)
    return accuracy_score(y_true, y_pred)

def safe_auroc(y_true, y_score):
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan

def safe_auprc(y_true, y_score):
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return average_precision_score(y_true, y_score)
    except Exception:
        return np.nan

In [6]:
rows = []

for g in group_order:
    model = models[g]
    y_true = Y_lock[group_to_label[g]].values
    y_prob = model.predict_proba(X_lock)[:, 1]

    rows.append({
        "disorder": g,
        "lockbox_AUROC": safe_auroc(y_true, y_prob),
        "lockbox_AUPRC": safe_auprc(y_true, y_prob),
        "lockbox_ACC": safe_acc(y_true, y_prob),
    })

results_df = pd.DataFrame(rows)
results_df

,disorder,lockbox_AUROC,lockbox_AUPRC,lockbox_ACC
0,DSI,0.969841,0.829195,0.950207
1,PHQ,0.963259,0.877887,0.904564
2,GAD,0.922483,0.893985,0.904564
3,PCL,0.909240,0.864840,0.842324
4,PDSS,0.977439,0.903358,0.954357
5,AUDIT,0.942278,0.828768,0.871369


In [ ]:
results_df.to_csv("lockbox_metrics_reproduced.csv", index=False)